### MOST OF THIS IS EXPERIMENTATION UNTIL THE COMPLETE CODE AT THE BOTTOM

In [2]:
# from pathlib import Path, PureWindowsPath # For windows only 
import os 
import pandas as pd
import time
from time import sleep
import selenium #Need this step 
from selenium import webdriver #Need this step 

from selenium.webdriver.common.by import By #Allows for selenium to click things 
from selenium.webdriver.chrome.service import Service #https://stackoverflow.com/questions/64717302/deprecationwarning-executable-path-has-been-deprecated-selenium-python
from selenium.webdriver.support import expected_conditions as EC #Allows for more complex code 
from selenium.webdriver.chrome.options import Options #Allows you to change aspects of the browser

# Establish options we can change
chrome_options = Options() 

In [26]:
chrome_options.add_argument("--window-size=1900,1000") #pick a size and stick with it, it can change how code intreacts with the website.
driver = webdriver.Chrome(options = chrome_options) #establish driver

In [27]:
url = 'https://lolalytics.com/lol/tierlist/'
driver.get(url) # Get the url

In [ ]:
# Source - https://stackoverflow.com/a/27760083
# Posted by OWADVL, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-15, License - CC BY-SA 4.0

driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # scrolls to the very bottom of the page to load everything (takes ~20 seconds)
time.sleep(5) # want to wait 10 seconds to let things load
driver.execute_script("window.scrollBy(0, -6000);") # scrolls back up to load everything
for i in range(0,20): # scrolls downs for the next twenty seconds to make sure everything loads
    driver.execute_script("window.scrollBy(0, 300);")
    time.sleep(1)
driver.execute_script("window.scrollTo(0,0);")

In [50]:
buckets = driver.find_elements(By.XPATH, '/html/body/main/div[6]/div') # creating bucket of each thing
del buckets[:2]
buckets[160].text

'71\nRyze\nC\n79.80\n48.98\n+1.44\n3.51\n0.93\n-9\n35,569\n49\n56.06\n2,269\n7.08\nGM'

In [14]:
driver.find_elements(By.XPATH, '/html/body/main/div[6]/div[3]/div[5]/div/img')[0].get_attribute('alt')

'top lane'

In [52]:
import time            # importing time package
start=time.time()      # start time

out = []
count = 3

for bucket in buckets:
    champion = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[3]/a')[0].text
    wr = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[6]/div/span')[0].text
    
    temp= driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[6]/div/span')
    wrdelta= temp[1].text if len(temp) > 1 else 0 # checks to see if there is an entry for wrdelta (some are missing on the website)

    pick = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[7]')[0].text
    ban = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[8]')[0].text
    PBI = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[9]')[0].text
    lane = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[5]/div/img')[0].get_attribute('alt')
    
    data = {
        'Champion':champion,
        'wr':wr,
        'wrdelta':wrdelta,
        'Pick Rate':pick,
        'Ban Rate':ban,
        'PBI Index':PBI,
        'lane':lane
    }
    out.append(data)
    count = count+1

end=time.time()        # end time

total_time=end-start   # measures total time by subtracting start by end
print(f' The code takes {round(total_time,5)} seconds to run.')

 The code takes 34.76131 seconds to run.


In [25]:
outdf=pd.DataFrame(out)
outdf

,Champion,wr,wrdelta,Pick Rate,Ban Rate,PBI Index,lane
0,Olaf,53.48,+1.66,2.35,2.24,6,top lane
1,Jinx,53.37,+1.11,14.38,4.57,34,bottom lane
2,Bard,51.64,+0.79,6.68,3.10,4,support lane
3,Naafiri,52.07,+1.40,7.12,22.15,9,jungle lane
4,Shen,52.81,+1.35,4.38,2.69,8,top lane
...,...,...,...,...,...,...,...
167,K'Sante,48.60,+1.02,3.88,2.32,-10,top lane
168,Jayce,49.37,+0.81,5.26,5.96,-10,top lane
169,Heimerdinger,50.54,+1.06,1.13,1.62,-1,top lane
170,Mel,45.43,+1.33,4.60,36.71,-41,middle lane


In [26]:
outdf['wrdelta']=pd.to_numeric(outdf['wrdelta'])
outdf['wr']=pd.to_numeric(outdf['wr'])

In [27]:
outdf['avgwr']=outdf['wr']-outdf['wrdelta']
outdf

,Champion,wr,wrdelta,Pick Rate,Ban Rate,PBI Index,lane,avgwr
0,Olaf,53.48,1.66,2.35,2.24,6,top lane,51.82
1,Jinx,53.37,1.11,14.38,4.57,34,bottom lane,52.26
2,Bard,51.64,0.79,6.68,3.10,4,support lane,50.85
3,Naafiri,52.07,1.40,7.12,22.15,9,jungle lane,50.67
4,Shen,52.81,1.35,4.38,2.69,8,top lane,51.46
...,...,...,...,...,...,...,...,...
167,K'Sante,48.60,1.02,3.88,2.32,-10,top lane,47.58
168,Jayce,49.37,0.81,5.26,5.96,-10,top lane,48.56
169,Heimerdinger,50.54,1.06,1.13,1.62,-1,top lane,49.48
170,Mel,45.43,1.33,4.60,36.71,-41,middle lane,44.10


In [43]:
driver.find_elements(By.XPATH,'/html/body/main/div[1]/div/div/div[3]/div/div/div[1]/img')[0].click() # clicks the rank sort
driver.find_elements(By.XPATH,'/html/body/main/div[1]/div/div/div[3]/div[2]/a[18]/div')[0].click() # clicks "challenger" rank

In [40]:
driver.find_elements(By.XPATH,'/html/body/main/div[1]/div/div/div[3]/div/div/div[1]/img')[0].click() # clicks the rank sort

In [41]:
driver.find_elements(By.XPATH,'/html/body/main/div[1]/div/div/div[3]/div[2]/a[8]/div')[0].click() # clicks "challenger" rank

### COMBINED EVERYTHING I'VE DONE INTO ONE BLOCK, THIS IS THE MAIN THING TO PAY ATTENTION TO !!!

In [ ]:
# from pathlib import Path, PureWindowsPath # For windows only 
import os 
import pandas as pd
import time
from time import sleep
import selenium #Need this step 
from selenium import webdriver #Need this step 

from selenium.webdriver.common.by import By #Allows for selenium to click things 
from selenium.webdriver.chrome.service import Service #https://stackoverflow.com/questions/64717302/deprecationwarning-executable-path-has-been-deprecated-selenium-python
from selenium.webdriver.support import expected_conditions as EC #Allows for more complex code 
from selenium.webdriver.chrome.options import Options #Allows you to change aspects of the browser

# Establish options we can change
chrome_options = Options() 

In [ ]:
# ranknum=[3, 4, 6, 8, 11, 13, 15, 17, 18, 19, 15] all ranks

In [ ]:
# ranknum=[3, 6, 8, 15, 18]

In [54]:
chrome_options.add_argument("--window-size=1900,1000")
driver = webdriver.Chrome(options = chrome_options) # establish driver

url = 'https://lolalytics.com/lol/tierlist/' # dataset for each champions
driver.get(url) # Get the url

ranknum=[3, 4, 6, 8, 11, 13, 15, 17, 18, 19] # the element for each rank selection

import time            # importing time package
start=time.time()      # start time

out2 = [] # empty list to convert into dataframe later

for r in ranknum:
    out = [] # empty list for data
    count = 3 # starts at 3 to skip the first 2 which are irrelevant elements
    driver.find_elements(By.XPATH,'/html/body/main/div[1]/div/div/div[3]/div/div/div[1]/img')[0].click() # clicks the rank sort
    time.sleep(2)
    driver.find_elements(By.XPATH,f'/html/body/main/div[1]/div/div/div[3]/div[2]/a[{r}]/div')[0].click() # clicks the rank
    time.sleep(1)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # scrolls to the very bottom of the page to load everything (takes ~20 seconds)
    time.sleep(5) # want to wait 5 seconds to let things load
    driver.execute_script("window.scrollBy(0, -6000);") # scrolls back up to load everything
    for i in range(0,18): # scrolls downs for the next 15 seconds to make sure everything loads
        driver.execute_script("window.scrollBy(0, 320);")
        time.sleep(1)
    driver.execute_script("window.scrollTo(0,0);") # scrolls back to the top of the page to also open up the rank selection menu for once the loop resets
    
    buckets = driver.find_elements(By.XPATH, '/html/body/main/div[6]/div') # creating a bucket of each champion
    del buckets[:2] # removes the first 2 from the list to skip the first 2
    
    rank = driver.find_elements(By.XPATH, '/html/body/main/div[1]/div/div/div[3]/div/div/div[2]')[0].text # grabs the rank

    for bucket in buckets: # runs the below code for every champion in the list
        champion = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[3]/a')[0].text # grabs champion name
        wr = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[6]/div/span')[0].text # grabs win rate
    
        temp= driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[6]/div/span')
        wrdelta= temp[1].text if len(temp) > 1 else 0 # checks to see if there is an entry for wrdelta (some are missing on the website)

        pick = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[7]')[0].text # grabs pick rate
        ban = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[8]')[0].text # grabs ban rate
        PBI = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[9]')[0].text # grabs PBI
        lane = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[5]/div/img')[0].get_attribute('alt') # grabs lane
        games = driver.find_elements(By.XPATH, f'/html/body/main/div[6]/div[{count}]/div[10]')[0].text # grabs the number of games

        wr=float(wr)
        wrdelta=float(wrdelta)
        PBI=float(PBI)

        avgwr=wr-wrdelta # gives the winrate of all tiers

        PBIno=PBI/wrdelta if wrdelta!=0 else PBI # removes win rate from the PBI
    
        data = {
            'Champion':champion,
            'wr':wr,
            'wrdelta':wrdelta,
            'avgwr':avgwr,
            'Pick Rate':pick,
            'Ban Rate':ban,
            'PBI Index':PBI,
            'PBIno':PBIno,
            'lane':lane,
            'games':games,
            'rank':rank
        }
        out.append(data) # adds champion data to a list that will be added at the end of this for loop
        count+=1
    out2.append(out) # EACH OF THE RANKS'S DATA IS ADDED TO A FINAL LIST THAT CAN BE SLICED TO GRAB EACH SET
    ## EDIT: STILL KEEPING THIS FUNCTION, BUT PUTTING EVERYTHING INTO ONE DATASET INSTEAD OF 10. STILL KEEPING JUST IN CASE !!!

end=time.time()        # end time

total_time=end-start   # measures total time by subtracting start by end
print(f' The code takes {round(total_time,5)} seconds to run.')

 The code takes 620.61902 seconds to run.


In [53]:
count

73

In [57]:
league=pd.DataFrame() # making an empty dataframe
c=0 # counter
for rank in out2:
    temp = pd.DataFrame(out2[c]) # slicing to turn each entry into a new dataframe to concat later
    league = pd.concat([league, temp], ignore_index=True)
    c+=1
league.to_csv('league.csv', index=False)

In [60]:
league.iloc[624]

Champion          Morgana
wr                  52.69
wrdelta              3.05
avgwr               49.64
Pick Rate            3.38
Ban Rate            14.23
PBI Index             0.0
PBIno                 0.0
lane         support lane
games              22,646
rank              DIAMOND
Name: 624, dtype: object

### ANYTHING PAST HERE IS LEGACY

In [17]:
challenger=out2[0]
gm=out2[1]
master=out2[2]
diamond=out2[3]
emerald=out2[4]
platinum=out2[5]
gold=out2[6]
silver=out2[7]
bronze=out2[8]
iron=out2[9]
league=out2[10]

In [22]:
challengerdf=pd.DataFrame(challenger)
gmdf=pd.DataFrame(gm)
masterdf=pd.DataFrame(master)
diamonddf=pd.DataFrame(diamond)
emeralddf=pd.DataFrame(emerald)
platinumdf=pd.DataFrame(platinum)
golddf=pd.DataFrame(gold)
silverdf=pd.DataFrame(silver)
bronzedf=pd.DataFrame(bronze)
irondf=pd.DataFrame(iron)
league=pd.DataFrame(league)

In [23]:
bronzedf

,Champion,wr,wrdelta,avgwr,Pick Rate,Ban Rate,PBI Index,PBIno,lane,games
0,Veigar,50.74,-2.76,53.50,9.89,7.19,33.0,-11.956522,middle lane,"100,629"
1,Sion,50.20,-2.86,53.06,2.97,1.28,8.0,-2.797203,top lane,"30,191"
2,Taric,50.81,-2.65,53.46,1.17,0.39,4.0,-1.509434,support lane,"11,867"
3,Shyvana,51.86,-2.60,54.46,6.51,7.45,29.0,-11.153846,jungle lane,"66,234"
4,Singed,51.21,-2.58,53.79,1.66,0.90,6.0,-2.325581,top lane,"16,941"
...,...,...,...,...,...,...,...,...,...,...
167,Gnar,44.95,-2.75,47.70,2.25,0.71,-6.0,2.181818,top lane,"22,866"
168,Camille,42.08,-2.51,44.59,0.96,0.43,-5.0,1.992032,top lane,"9,810"
169,Mel,46.19,-1.99,48.18,7.82,44.51,-21.0,10.552764,middle lane,"79,600"
170,Azir,42.47,-1.96,44.43,1.68,0.51,-9.0,4.591837,middle lane,"17,119"


In [25]:
challengerdf.to_csv('challenger.csv', index=False)
gmdf.to_csv('gm.csv', index=False)
masterdf.to_csv('master.csv', index=False)
diamonddf.to_csv('diamond.csv', index=False)
emeralddf.to_csv('emerald.csv', index=False)
platinumdf.to_csv('platinum.csv', index=False)
golddf.to_csv('gold.csv', index=False)
silverdf.to_csv('silver.csv', index=False)
bronzedf.to_csv('bronze.csv', index=False)
irondf.to_csv('iron.csv', index=False)

In [3]:
league=pd.read_csv(r"C:\Users\ong92\Documents\GitHub\ECO 590\Class-Notes\league.csv")

In [6]:
league['PBInowr']=league['PBI Index']/(league['wrdelta'])

In [7]:
league

,Champion,wr,wrdelta,Pick Rate,Ban Rate,PBI Index,lane,avgwr,PBInowr
0,Olaf,53.48,1.66,2.35,2.24,6,top lane,51.82,3.614458
1,Jinx,53.37,1.11,14.38,4.57,34,bottom lane,52.26,30.630631
2,Bard,51.64,0.79,6.68,3.10,4,support lane,50.85,5.063291
3,Naafiri,52.07,1.40,7.12,22.15,9,jungle lane,50.67,6.428571
4,Shen,52.81,1.35,4.38,2.69,8,top lane,51.46,5.925926
...,...,...,...,...,...,...,...,...,...
167,K'Sante,48.60,1.02,3.88,2.32,-10,top lane,47.58,-9.803922
168,Jayce,49.37,0.81,5.26,5.96,-10,top lane,48.56,-12.345679
169,Heimerdinger,50.54,1.06,1.13,1.62,-1,top lane,49.48,-0.943396
170,Mel,45.43,1.33,4.60,36.71,-41,middle lane,44.10,-30.827068
